In [17]:
import numpy as np
import pandas as pd
import os

os.chdir("/Users/acastano/Desktop/crispr_carT_AI_analysis")

# If needed, re-load metadata
metadata = pd.read_csv("data/processed/metadata_samples.csv", index_col=0)

# Library sizes and log2CPM
lib_sizes = counts_num.sum(axis=0)
cpm = counts_num.divide(lib_sizes, axis=1) * 1e6
expr_logcpm = np.log2(cpm + 1)

expr_logcpm.shape



(60675, 60)

In [18]:
#DE helper function
from scipy import stats
from statsmodels.stats.multitest import multipletests

def differential_expression(expr_df, metadata, mask_group1, mask_group2,
                            group1_name="group1", group2_name="group2"):
    samples1 = metadata.index[mask_group1]
    samples2 = metadata.index[mask_group2]
    
    X1 = expr_df.loc[:, samples1]
    X2 = expr_df.loc[:, samples2]
    
    mean1 = X1.mean(axis=1)
    mean2 = X2.mean(axis=1)
    log2_fc = mean2 - mean1
    
    t_stats, pvals = stats.ttest_ind(
        X2.T, X1.T,
        equal_var=False,
        nan_policy="omit"
    )
    
    _, pval_adj, _, _ = multipletests(pvals, method="fdr_bh")
    
    return pd.DataFrame({
        f"mean_{group1_name}": mean1,
        f"mean_{group2_name}": mean2,
        "log2_fc": log2_fc,
        "pval": pvals,
        "pval_adj": pval_adj
    })


In [19]:
#DE: Late (≥168h) vs Early (0h)
early_mask = metadata["hours"] == 0
late_mask  = metadata["hours"] >= 168

de_late_vs_early = differential_expression(
    expr_logcpm, metadata,
    mask_group1=early_mask,
    mask_group2=late_mask,
    group1_name="early_0h",
    group2_name="late_168hplus"
)

de_late_vs_early.head()


,mean_early_0h,mean_late_168hplus,log2_fc,pval,pval_adj
gene,,,,,
ENSG00000223972,0.055567,0.049877,-0.005689,0.811482,NaN
ENSG00000227232,2.649494,2.853152,0.203658,0.187170,NaN
ENSG00000278267,0.242513,0.366924,0.124411,0.025860,NaN
ENSG00000243485,0.018491,0.013329,-0.005163,0.801665,NaN
ENSG00000284332,0.000000,0.000000,0.000000,NaN,NaN


In [20]:
#DE: RHOG vs SafeHarbor at 0h
mask_0h  = metadata["hours"] == 0
mask_sh  = mask_0h & (metadata["guide"] == "SafeHarbor")
mask_rhg = mask_0h & (metadata["guide"] == "RHOG")

de_rhog_vs_safe_0h = differential_expression(
    expr_logcpm, metadata,
    mask_group1=mask_sh,
    mask_group2=mask_rhg,
    group1_name="SafeHarbor_0h",
    group2_name="RHOG_0h"
)

de_rhog_vs_safe_0h.head()


,mean_SafeHarbor_0h,mean_RHOG_0h,log2_fc,pval,pval_adj
gene,,,,,
ENSG00000223972,0.025015,0.086119,0.061104,0.153861,NaN
ENSG00000227232,2.522880,2.776109,0.253228,0.358706,NaN
ENSG00000278267,0.242380,0.242646,0.000266,0.997646,NaN
ENSG00000243485,0.000000,0.036982,0.036982,0.363217,NaN
ENSG00000284332,0.000000,0.000000,0.000000,NaN,NaN


In [21]:
#SAVE the DE tables to results/
os.makedirs("results", exist_ok=True)

de_late_vs_early.to_csv("results/de_late_vs_early.csv")
de_rhog_vs_safe_0h.to_csv("results/de_rhog_vs_safe_0h.csv")

!ls -l results


total 18464
-rw-r--r--@ 1 acastano  staff  4373236 Nov 28 16:34 de_late_vs_early.csv
-rw-r--r--@ 1 acastano  staff  3971182 Nov 28 16:34 de_rhog_vs_safe_0h.csv
drwxr-xr-x  2 acastano  staff       64 Nov 28 13:38 pathways


In [22]:
de_late_vs_early.head()


,mean_early_0h,mean_late_168hplus,log2_fc,pval,pval_adj
gene,,,,,
ENSG00000223972,0.055567,0.049877,-0.005689,0.811482,NaN
ENSG00000227232,2.649494,2.853152,0.203658,0.187170,NaN
ENSG00000278267,0.242513,0.366924,0.124411,0.025860,NaN
ENSG00000243485,0.018491,0.013329,-0.005163,0.801665,NaN
ENSG00000284332,0.000000,0.000000,0.000000,NaN,NaN


In [23]:
de_late_vs_early.isna().sum()


mean_early_0h             0
mean_late_168hplus        0
log2_fc                   0
pval                  20919
pval_adj              60675
dtype: int64

In [24]:
import numpy as np
import pandas as pd

counts = pd.read_csv("data/raw/GSE266618_counts.csv", index_col=0)

# Just to be sure they’re numeric (as you showed)
counts_num = counts.apply(pd.to_numeric, errors="coerce")
print("NaNs in counts_num:", counts_num.isna().sum().sum())

lib_sizes = counts_num.sum(axis=0)
cpm = counts_num.divide(lib_sizes, axis=1) * 1e6
expr_logcpm = np.log2(cpm + 1)


NaNs in counts_num: 0


In [25]:
from scipy import stats
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd

def differential_expression_safe(expr_df, metadata,
                                 mask_group1, mask_group2,
                                 group1_name="group1", group2_name="group2"):
    samples1 = metadata.index[mask_group1]
    samples2 = metadata.index[mask_group2]

    X1 = expr_df.loc[:, samples1]
    X2 = expr_df.loc[:, samples2]

    mean1 = X1.mean(axis=1)
    mean2 = X2.mean(axis=1)
    log2_fc = mean2 - mean1

    # Welch t-test across genes
    t_stat, pvals = stats.ttest_ind(
        X2.T, X1.T,
        equal_var=False,
        nan_policy="omit"
    )

    pvals = pd.Series(pvals, index=expr_df.index, name="pval")

    # Allocate full FDR vector with NaNs
    pval_adj = pd.Series(np.nan, index=pvals.index, name="pval_adj")

    valid = pvals.notna()
    if valid.sum() > 0:
        _, pval_adj_valid, _, _ = multipletests(
            pvals[valid].values,
            method="fdr_bh"
        )
        pval_adj.loc[valid] = pval_adj_valid

    res = pd.DataFrame({
        f"mean_{group1_name}": mean1,
        f"mean_{group2_name}": mean2,
        "log2_fc": log2_fc,
        "pval": pvals,
        "pval_adj": pval_adj
    })

    return res


In [26]:
# Late vs early
early_mask = metadata["hours"] == 0
late_mask  = metadata["hours"] >= 168

de_late_vs_early = differential_expression_safe(
    expr_logcpm, metadata,
    early_mask, late_mask,
    group1_name="early_0h",
    group2_name="late_168hplus"
)

de_late_vs_early.isna().sum()


mean_early_0h             0
mean_late_168hplus        0
log2_fc                   0
pval                  20919
pval_adj              20919
dtype: int64

In [27]:
# RHOG vs SafeHarbor at 0h
mask_0h  = metadata["hours"] == 0
mask_sh  = mask_0h & (metadata["guide"] == "SafeHarbor")
mask_rhg = mask_0h & (metadata["guide"] == "RHOG")

de_rhog_vs_safe_0h = differential_expression_safe(
    expr_logcpm, metadata,
    mask_sh, mask_rhg,
    group1_name="SafeHarbor_0h",
    group2_name="RHOG_0h"
)

de_rhog_vs_safe_0h.isna().sum()


mean_SafeHarbor_0h        0
mean_RHOG_0h              0
log2_fc                   0
pval                  26693
pval_adj              26693
dtype: int64

In [28]:
def make_gene_sets(de_df, logfc_col="log2_fc", fdr_col="pval_adj",
                   logfc_thresh=0.5, fdr_thresh=0.2):
    up = de_df.loc[
        (de_df[logfc_col] >= logfc_thresh) &
        (de_df[fdr_col] <= fdr_thresh)
    ].sort_values(logfc_col, ascending=False)

    down = de_df.loc[
        (de_df[logfc_col] <= -logfc_thresh) &
        (de_df[fdr_col] <= fdr_thresh)
    ].sort_values(logfc_col, ascending=True)

    up_genes = up.index.tolist()
    down_genes = down.index.tolist()
    ranked = de_df[logfc_col].sort_values(ascending=False)

    return up_genes, down_genes, ranked

up_late, down_late, ranks_late = make_gene_sets(de_late_vs_early)
up_rhog, down_rhog, ranks_rhog = make_gene_sets(de_rhog_vs_safe_0h)

len(up_late), len(down_late), len(up_rhog), len(down_rhog)


(2521, 2669, 0, 0)

In [29]:
os.makedirs("data/processed", exist_ok=True)
expr_logcpm.to_parquet("data/processed/expr_logcpm.parquet")


In [30]:
#Safe DE function (handles NaNs in p-values)
from scipy import stats
from statsmodels.stats.multitest import multipletests

def differential_expression_safe(expr_df, metadata,
                                 mask_group1, mask_group2,
                                 group1_name="group1", group2_name="group2"):
    samples1 = metadata.index[mask_group1]
    samples2 = metadata.index[mask_group2]

    X1 = expr_df.loc[:, samples1]
    X2 = expr_df.loc[:, samples2]

    mean1 = X1.mean(axis=1)
    mean2 = X2.mean(axis=1)
    log2_fc = mean2 - mean1

    # Welch t-test
    t_stat, pvals = stats.ttest_ind(
        X2.T, X1.T,
        equal_var=False,
        nan_policy="omit"
    )

    pvals = pd.Series(pvals, index=expr_df.index, name="pval")

    # FDR only on non-NaN pvals
    pval_adj = pd.Series(np.nan, index=pvals.index, name="pval_adj")
    valid = pvals.notna()
    if valid.sum() > 0:
        _, pval_adj_valid, _, _ = multipletests(
            pvals[valid].values,
            method="fdr_bh"
        )
        pval_adj.loc[valid] = pval_adj_valid

    res = pd.DataFrame({
        f"mean_{group1_name}": mean1,
        f"mean_{group2_name}": mean2,
        "log2_fc": log2_fc,
        "pval": pvals,
        "pval_adj": pval_adj
    })

    return res


In [31]:
#Recompute both DE contrasts from this clean matrix
early_mask = metadata["hours"] == 0
late_mask  = metadata["hours"] >= 168

de_late_vs_early = differential_expression_safe(
    expr_logcpm, metadata,
    early_mask, late_mask,
    group1_name="early_0h",
    group2_name="late_168hplus"
)

de_late_vs_early.isna().sum()
de_late_vs_early.head()


,mean_early_0h,mean_late_168hplus,log2_fc,pval,pval_adj
gene,,,,,
ENSG00000223972,0.055567,0.049877,-0.005689,0.811482,0.870327
ENSG00000227232,2.649494,2.853152,0.203658,0.187170,0.349364
ENSG00000278267,0.242513,0.366924,0.124411,0.025860,0.075007
ENSG00000243485,0.018491,0.013329,-0.005163,0.801665,0.862357
ENSG00000284332,0.000000,0.000000,0.000000,NaN,NaN


In [32]:
#RHOG vs SafeHarbor (0h)
mask_0h  = metadata["hours"] == 0
mask_sh  = mask_0h & (metadata["guide"] == "SafeHarbor")
mask_rhg = mask_0h & (metadata["guide"] == "RHOG")

de_rhog_vs_safe_0h = differential_expression_safe(
    expr_logcpm, metadata,
    mask_sh, mask_rhg,
    group1_name="SafeHarbor_0h",
    group2_name="RHOG_0h"
)

de_rhog_vs_safe_0h.isna().sum()
de_rhog_vs_safe_0h.head()


,mean_SafeHarbor_0h,mean_RHOG_0h,log2_fc,pval,pval_adj
gene,,,,,
ENSG00000223972,0.025015,0.086119,0.061104,0.153861,0.903430
ENSG00000227232,2.522880,2.776109,0.253228,0.358706,0.903430
ENSG00000278267,0.242380,0.242646,0.000266,0.997646,0.999558
ENSG00000243485,0.000000,0.036982,0.036982,0.363217,0.903430
ENSG00000284332,0.000000,0.000000,0.000000,NaN,NaN


In [33]:
os.makedirs("results", exist_ok=True)
de_late_vs_early.to_csv("results/de_late_vs_early.csv")
de_rhog_vs_safe_0h.to_csv("results/de_rhog_vs_safe_0h.csv")


In [34]:
#Build gene sets with realistic thresholds for bulk RNA-seq
def make_gene_sets(de_df, logfc_col="log2_fc", fdr_col="pval_adj",
                   logfc_thresh=0.5, fdr_thresh=0.2):
    up = de_df.loc[
        (de_df[logfc_col] >= logfc_thresh) &
        (de_df[fdr_col] <= fdr_thresh)
    ].sort_values(logfc_col, ascending=False)

    down = de_df.loc[
        (de_df[logfc_col] <= -logfc_thresh) &
        (de_df[fdr_col] <= fdr_thresh)
    ].sort_values(logfc_col, ascending=True)

    up_genes = up.index.tolist()
    down_genes = down.index.tolist()
    ranked = de_df[logfc_col].sort_values(ascending=False)

    return up_genes, down_genes, ranked

up_late, down_late, ranks_late = make_gene_sets(de_late_vs_early)
up_rhog, down_rhog, ranks_rhog = make_gene_sets(de_rhog_vs_safe_0h)

len(up_late), len(down_late), len(up_rhog), len(down_rhog)



(2521, 2669, 0, 0)

In [1]:
import os

print("CWD:", os.getcwd())
print("results/pca contents:", os.listdir("results/pca"))


CWD: /Users/acastano/Desktop/crispr_carT_AI_analysis/notebooks/Notebook_Utility


FileNotFoundError: [Errno 2] No such file or directory: 'results/pca'